## Detour reranking with the Routing API

This notebook implements detour reranking **from scratch**, on top of
`routing_api.RoutingAPI` only. Everything below is meant to be copied into an existing
application ranking layer; `detour.DetourRanker` is the equivalent implementation of this
demo, and is compared with it at the end.

The quantity to compute for each search result is its *excursion*, the extra distance the
driver pays to stop there:

```
excursion = dist(at → result) + dist(result → stop) − dist(at → stop)
```

The cell below rebuilds the route and the search of the
[Routing API notebook](obm_9_routing_api.ipynb):

In [ ]:
from flexpolyline import decode, encode

from here_search_demo.api import API
from here_search_demo.auth import Credentials
from here_search_demo.http import HTTPSession
from here_search_demo.routing_api import RoutingAPI, Waypoint
from here_search_demo.widgets.route_geometry import simplify_polyline

credentials = Credentials()
sent = []

api = API(credentials=credentials)
routing_api = RoutingAPI(credentials=credentials, on_request_sent=lambda request: sent.append(request))
session = HTTPSession()

start_pos = 48.57436, 7.71556  # Strasbourg, where the trip started
at_pos = 48.03529, 7.37121  # near Colmar, where the driver is now
stop_pos = 45.75049, 4.84418  # Lyon, the destination

route = await routing_api.routes(session=session, origin=start_pos, destination=stop_pos)
(route_section,) = RoutingAPI.sections(route)

# The detour baseline is the *remaining* journey, at → stop, not the whole route.
remaining = await routing_api.routes(session=session, origin=at_pos, destination=stop_pos)
(baseline,) = RoutingAPI.sections(remaining)

resp = await api.browse(
    session=session,
    latitude=at_pos[0],
    longitude=at_pos[1],
    categories=["100-1000"],
    polyline=encode(simplify_polyline(decode(route_section.polyline), 2000)),
    width=500,
    limit=20,
    lang="en",
)
resp.titles

### One via-route call per result

`at → via(result) → stop` returns **both** legs of a detour in a single call: `sections[0]`
is `at → result`, `sections[1]` is `result → stop`. Asking for `summary,polyline` returns
the durations and distances needed for the ranking *and* the geometry needed to draw the
detour later, so no second call is ever required.

In [ ]:
async def detour_legs(item):
    """Return the (at → item, item → stop) sections of one search result."""
    position = item["position"]
    response = await routing_api.routes(
        session=session,
        origin=at_pos,
        destination=stop_pos,
        via=Waypoint(position["lat"], position["lng"], item.get("address", {}).get("street")),
        return_fields=("summary", "polyline"),
    )
    to_leg, from_leg = RoutingAPI.sections(response)
    return to_leg, from_leg


item = resp.data["items"][0]
to_leg, from_leg = await detour_legs(item)

print("baseline at → stop:", baseline.duration, "s,", baseline.length, "m")
print("at → item:         ", to_leg.duration, "s,", to_leg.length, "m")
print("item → stop:       ", from_leg.duration, "s,", from_leg.length, "m")
print("excursion:         ", to_leg.length + from_leg.length - baseline.length, "m")

### Reranking a response

All the calls are independent, so they are fired in parallel with `asyncio.gather()`. Only
positioned items are considered: a query-suggestion or a chain-suggestion has no
`position` and must be left where it is.

The results are sorted by `dur_to`, the time needed to *reach* them from the current
position. `dur_from` is not part of the sort key, but it is kept: it is what makes the
excursion — and the total round-trip time — displayable next to each result.

The rank the item held in the original, distance-ordered response is captured **before**
the sort and kept as `rank_before`. It costs nothing and it is what lets the interface
justify itself: a result that jumps from the bottom of the list to the top is the whole
point of the feature, and showing where it came from keeps that legible instead of looking
like a shuffle.

In [ ]:
import asyncio


def annotate(item, rank_before, to_leg, from_leg, baseline):
    excursion = to_leg.length + from_leg.length - baseline.length
    return dict(
        item,
        _detour={
            "rank_before": rank_before,  # where the item stood in the distance-ordered response
            "dur_to_sec": to_leg.duration,
            "dur_from_sec": from_leg.duration,
            "excursion_m": excursion,
            "polyline_to": to_leg.polyline,
            "polyline_from": from_leg.polyline,
            "label": f"{item['title']} — {round(to_leg.duration / 60)} min, {excursion / 1000:+.1f} km",
        },
    )


async def rerank(items, max_excursion=None):
    """Return the positioned items sorted by driving time, annotated with their detour."""
    positioned = [(rank, item) for rank, item in enumerate(items) if "position" in item]
    legs = await asyncio.gather(*(detour_legs(item) for _, item in positioned))

    ranked = [
        annotate(item, rank, to_leg, from_leg, baseline)
        for (rank, item), (to_leg, from_leg) in zip(positioned, legs)
    ]
    if max_excursion is not None:
        ranked = [item for item in ranked if item["_detour"]["excursion_m"] <= max_excursion]

    ranked.sort(key=lambda item: item["_detour"]["dur_to_sec"])
    return ranked


calls_before = len(sent)
ranked = await rerank(resp.data["items"])

print(len(sent) - calls_before, "routing requests for", len(resp.data["items"]), "items")
for item in ranked:
    print(item["_detour"]["label"])

One request less than the number of items: the first via-route was already fetched above
and served from the `RoutingAPI` cache. The `at → stop` baseline was fetched once, up
front; as long as the driver stays there, every later reranking reuses it.

The annotation is deliberately kept in a private `_detour` key of a **copy** of the item.
Nothing is mutated, so the original response can be re-displayed as-is when the user turns
the travel-time ranking off — without a single new request.

In [ ]:
template = "{:>4} {:>6} {:<26} {:>9} {:>11} {:>12} {:>12}"
print(template.format("rank", "was", "title", "to (min)", "from (min)", "excursion", "round-trip"))
for rank, item in enumerate(ranked):
    detour = item["_detour"]
    moved = detour["rank_before"] - rank
    print(
        template.format(
            rank,
            f"#{detour['rank_before']} {'↑' if moved > 0 else '↓' if moved < 0 else '='}",
            item["title"][:25],
            round(detour["dur_to_sec"] / 60),
            round(detour["dur_from_sec"] / 60),
            f"{detour['excursion_m']}m",
            round((detour["dur_to_sec"] + detour["dur_from_sec"]) / 60),
        )
    )

Look at the `was` column: the corridor search returned these places ordered
by straight-line distance, and driving time reshuffles them heavily. A place a few hundred
meters away on the other side of a motorway sinks; one further away but on the right side of
the carriageway, with a direct exit, rises to the top. That reordering *is* the feature, and
`rank_before` lets an interface show it — a "was #12" badge, an arrow, or a
side-by-side of the two orders.

### Dropping the results that are too far off

Filtering is equally cheap, because it works on data that is already cached: `max_excursion`
drops the results whose excursion exceeds the detour the user is willing to pay. A common
choice is a small multiple of the corridor width.

In [ ]:
calls_before = len(sent)

for max_excursion in (None, 10000, 3000, 2000, 1000):
    filtered = await rerank(resp.data["items"], max_excursion=max_excursion)
    print(f"max_excursion={str(max_excursion):>5}: {len(filtered):>2} items")

print("extra routing requests:", len(sent) - calls_before)

### The pattern, in six points

Everything above boils down to a short recipe:

1. keep one `RoutingAPI` instance (or your own routing client) with a **shared cache**;
2. fetch the route once for the search corridor, and the `at → stop` leg once for the
   detour baseline — both stay valid as long as the driver has not moved;
3. per result, one `at → via(result) → stop` call with `return=summary,polyline`, all of
   them gathered in parallel;
4. annotate copies of the items, never the response itself, keeping the pre-rerank rank;
5. sort by `dur_to`, and optionally filter on `excursion_m`;
6. keep the two returned polylines: rendering the detour then costs nothing.

### The same recipe, packaged

`detour.DetourRanker` implements exactly those six points, over a cache shared with
`RouteEngine`. Its `rerank()` sits one level up from the one written above: it takes the
whole `Response` rather than its items, and returns a new `Response` rather than a list.
The annotation carries the same keys, `rank_before` included, plus `dist_to_m` and
`dist_from_m`, and formats its `label` differently. The cache is already warm, so the call
below sends no request and reproduces the order obtained by hand:


In [ ]:
from here_search_demo.detour import DetourRanker

ranker = DetourRanker(credentials=credentials, at_pos=at_pos, stop_pos=stop_pos, routing_api=routing_api)
reranked = await ranker.rerank(resp)

for rank, item in enumerate(reranked.data["items"]):
    print(f"{rank:>3}  was #{item['_detour']['rank_before']:<3} {item['_detour']['label']}")

print()
print("same order as above:", [item["_detour"]["rank_before"] for item in reranked.data["items"]]
      == [item["_detour"]["rank_before"] for item in ranked])

The [next notebook](obm_13_detour_render.ipynb) draws those detours on a map, without any
further Routing API call.

<!--
from urllib.parse import quote, unquote
svg = """<svg width="40" height="48" viewBox="0 0 40 48" fill="none" xmlns="http://www.w3.org/2000/svg"><path class="triangle" d="M11,36.8l-5.5,5.5L0,36.8H11z" fill="#48DAD0"></path><path class="HERE" d="M19.1,24.2c-1.2-1.4-1.1-2.1-0.4-2.8c0.9-0.9,1.7-0.5,2.7,0.5L19.1,24.2z M30.9,9.2c0.9-0.9,1.7-0.5,2.7,0.5
    L31.3,12C30.1,10.7,30.2,9.9,30.9,9.2z M38,11.6c-1.1,1.6-2.9,4.2-4.9,2.2l5-5c-0.4-0.5-0.8-0.9-1-1.1c-2.7-2.7-5.7-2.7-8-0.4
	c-1.6,1.6-2,3.4-1.5,5.1l-1.6-1.8c-0.5,0.3-2.4,1.9-0.9,4.5l-1.8-1.5l-2.4,2.4l3.2,3.2c-2.5-1.9-5.1-1.8-7.2,0.4
	c-2.3,2.3-2.1,5-0.4,7.3l-0.3-0.3c-2.3-2.3-4.7-1.5-5.9-0.3c-0.9,0.9-1.5,2.2-1.3,3.1L4,24.6l-2.6,2.6l9.6,9.6h5.2l-3.5-3.5
	c-1.8-1.8-1.8-2.8-1-3.7c0.8-0.8,1.8-0.3,3.6,1.4l3.4,3.4l2.6-2.6l-3.2-3.2c2.3,1.8,5.1,1.9,7.7-0.6l0,0c1.5-1.4,2-2.8,2-2.8
	l-1.9-1.3c-1.1,1.6-2.9,4.2-4.9,2.3l5-5l3.1,3.1l2.7-2.7l-3.9-3.9c-1.8-1.8-0.7-3.5,0-4.1c0.4,0.7,0.9,1.5,1.5,2.1
	c2.5,2.5,5.7,3,8.6,0.2l0,0c1.5-1.4,2-2.8,2-2.8S38,11.6,38,11.6z" fill="#000000"></path></svg>
"""
print(f"""![HERE](data:image/svg+xml,{quote(svg)})
<span style="float:right; width:90%;"><sub><b>Copyright (c) 2020-2026 HERE Global B.V. and its affiliate(s). All rights reserved.</b>
This software, including documentation, is protected by copyright controlled by HERE. All rights are reserved. Copying, including reproducing, 
storing, adapting or translating, any or all of this material requires the prior written consent of HERE. This material also contains confidential 
information which may not be disclosed to others without the prior written consent of HERE.</sub></span>""")
-->
![HERE](data:image/svg+xml,%3Csvg%20width%3D%2240%22%20height%3D%2248%22%20viewBox%3D%220%200%2040%2048%22%20fill%3D%22none%22%20xmlns%3D%22http%3A//www.w3.org/2000/svg%22%3E%3Cpath%20class%3D%22triangle%22%20d%3D%22M11%2C36.8l-5.5%2C5.5L0%2C36.8H11z%22%20fill%3D%22%2348DAD0%22%3E%3C/path%3E%3Cpath%20class%3D%22HERE%22%20d%3D%22M19.1%2C24.2c-1.2-1.4-1.1-2.1-0.4-2.8c0.9-0.9%2C1.7-0.5%2C2.7%2C0.5L19.1%2C24.2z%20M30.9%2C9.2c0.9-0.9%2C1.7-0.5%2C2.7%2C0.5%0A%20%20%20%20L31.3%2C12C30.1%2C10.7%2C30.2%2C9.9%2C30.9%2C9.2z%20M38%2C11.6c-1.1%2C1.6-2.9%2C4.2-4.9%2C2.2l5-5c-0.4-0.5-0.8-0.9-1-1.1c-2.7-2.7-5.7-2.7-8-0.4%0A%09c-1.6%2C1.6-2%2C3.4-1.5%2C5.1l-1.6-1.8c-0.5%2C0.3-2.4%2C1.9-0.9%2C4.5l-1.8-1.5l-2.4%2C2.4l3.2%2C3.2c-2.5-1.9-5.1-1.8-7.2%2C0.4%0A%09c-2.3%2C2.3-2.1%2C5-0.4%2C7.3l-0.3-0.3c-2.3-2.3-4.7-1.5-5.9-0.3c-0.9%2C0.9-1.5%2C2.2-1.3%2C3.1L4%2C24.6l-2.6%2C2.6l9.6%2C9.6h5.2l-3.5-3.5%0A%09c-1.8-1.8-1.8-2.8-1-3.7c0.8-0.8%2C1.8-0.3%2C3.6%2C1.4l3.4%2C3.4l2.6-2.6l-3.2-3.2c2.3%2C1.8%2C5.1%2C1.9%2C7.7-0.6l0%2C0c1.5-1.4%2C2-2.8%2C2-2.8%0A%09l-1.9-1.3c-1.1%2C1.6-2.9%2C4.2-4.9%2C2.3l5-5l3.1%2C3.1l2.7-2.7l-3.9-3.9c-1.8-1.8-0.7-3.5%2C0-4.1c0.4%2C0.7%2C0.9%2C1.5%2C1.5%2C2.1%0A%09c2.5%2C2.5%2C5.7%2C3%2C8.6%2C0.2l0%2C0c1.5-1.4%2C2-2.8%2C2-2.8S38%2C11.6%2C38%2C11.6z%22%20fill%3D%22%23000000%22%3E%3C/path%3E%3C/svg%3E%0A)
<span style="float:right; width:90%;"><sub><b>Copyright (c) 2020-2026 HERE Global B.V. and its affiliate(s). All rights reserved.</b>
This software, including documentation, is protected by copyright controlled by HERE. All rights are reserved. Copying, including reproducing, 
storing, adapting or translating, any or all of this material requires the prior written consent of HERE. This material also contains confidential 
information which may not be disclosed to others without the prior written consent of HERE.</sub></span>